##  Seed metadata (run once per environment)

Seeds `ict_metadata.etl_setup_config`, `ict_metadata.column_mapping` and `ict_metadata.etl_watermark` for each entity. Run after [DDL_Notebook](/Shared/gskglobal-ict/bundle_ict/Utilities/DDL_Notebook) has created the tables.

* Parameter: `catalog_name` (e.g. `ent_gf_dev`).
* Safe to re-run: `MERGE` statements keyed on each table's natural key.
  * `etl_setup_config` (`entity_id`, `target_layer`) and `column_mapping` (`table_name`, `target_layer`, `source_column`): new rows are inserted, existing rows are updated to the values below.
  * `etl_watermark` (`entity_id`, `layer`): only missing rows are inserted; existing watermark progress is never reset.

In [0]:
dbutils.widgets.text("catalog_name", "ent_gf_dev", "catalog_name")

In [0]:
%sql
MERGE INTO IDENTIFIER(:catalog_name || '.metadata_ict.etl_setup_config') AS t
USING (
  SELECT * FROM VALUES
    ('zipt_markup', 'raw', 'CB_TCK03', 'r_ict_nonconf.tck03', 'append', CAST(NULL AS ARRAY<STRING>), '', '_ingest_date', 1095, 1, TRUE),
    ('zipt_markup', 'raw', 'CB_ZIPT_MARKUP', 'r_ict_conf.zipt_markup', 'append', CAST(NULL AS ARRAY<STRING>), '', '_ingest_date', 1095, 1, TRUE),
    ('zipt_markup', 'foundation', 'r_ict_conf.zipt_markup', 'f_ict_conf.zipt_markup', 'append', CAST(NULL AS ARRAY<STRING>), '', '_ingest_date', 1095, 2, TRUE),
    ('zipt_markup', 'trusted', 'f_ict_conf.zipt_markup', 't_ict_conf.zipt_markup', 'merge', ARRAY('MANDT', 'MATNR', 'WERKS', 'ZBUKRS', 'PERIOD', 'ZYEAR', 'ZCATEGORY', 'ZDEP', 'ZEX_VALVAR', 'KURST', 'BWDAT'), 'SCD1', '_ingest_date', 1095, 3, TRUE),
    ('zipt_markup', 'unified', 't_ict_conf.zipt_markup', 'u_ict_conf.zipt_markup', 'merge', ARRAY('MANDT', 'MATERIAL', 'PLANT', 'COMP_CODE', 'FISCPER3', 'FISCYEAR', 'fs_bic_fs_ycategry', 'ZDEP', 'ZEX_VALVAR', 'RATE_TYPE', 'BWDAT'), 'SCD1', '_ingest_date', 1095, 4, TRUE)
  AS s(entity_id, target_layer, source_read_from, target_table, load_type, primary_key_cols, scd_type, partition_column, raw_retention_days, load_sequence, is_active)
) AS s
ON t.entity_id = s.entity_id AND t.target_layer = s.target_layer
WHEN MATCHED THEN UPDATE SET
  t.source_read_from   = s.source_read_from,
  t.target_table       = s.target_table,
  t.load_type          = s.load_type,
  t.primary_key_cols   = s.primary_key_cols,
  t.scd_type           = s.scd_type,
  t.partition_column   = s.partition_column,
  t.raw_retention_days = s.raw_retention_days,
  t.load_sequence      = s.load_sequence,
  t.is_active          = s.is_active,
  t.updated_ts         = current_timestamp(),
  t.updated_by         = current_user()
WHEN NOT MATCHED THEN INSERT
  (entity_id, target_layer, source_read_from, target_table, load_type, primary_key_cols, scd_type, partition_column, raw_retention_days, load_sequence, is_active, created_ts, created_by, updated_ts, updated_by)
VALUES
  (s.entity_id, s.target_layer, s.source_read_from, s.target_table, s.load_type, s.primary_key_cols, s.scd_type, s.partition_column, s.raw_retention_days, s.load_sequence, s.is_active, current_timestamp(), current_user(), NULL, NULL)

In [0]:
%sql
MERGE INTO IDENTIFIER(:catalog_name || '.metadata_ict.column_mapping') AS t
USING (
  SELECT * FROM VALUES
  ('zipt_markup','trusted','MATNR','MATERIAL','STRING',NULL,TRUE,TRUE),
  ('zipt_markup','trusted','WERKS','PLANT','STRING',NULL,TRUE,TRUE),
  ('zipt_markup','trusted','ZBUKRS','COMP_CODE','STRING',NULL,TRUE,TRUE),
  ('zipt_markup','trusted','PERIOD','FISCPER3','STRING',NULL,TRUE,TRUE),
  ('zipt_markup','trusted','ZYEAR','FISCYEAR','STRING',NULL,TRUE,TRUE),
  ('zipt_markup','trusted','MFINCAL','BBP_FLAG','STRING',NULL,FALSE,TRUE),
  ('zipt_markup','trusted','ZCATEGORY','fs_bic_fs_ycategry','STRING',NULL,TRUE,TRUE),
  ('zipt_markup','trusted','KURST','RATE_TYPE','STRING',NULL,TRUE,TRUE),
  ('zipt_markup','trusted','ZCURR','CURRENCY','STRING',NULL,FALSE,TRUE),
  ('zipt_markup','trusted','ZNETCO','fs_bic_fs_ynetco6','STRING',NULL,FALSE,TRUE),
  ('zipt_markup','trusted','ZINTPR','fs_bic_fs_yintpr6','STRING',NULL,FALSE,TRUE),
  ('zipt_markup','trusted','ZMARKUP','fs_bic_fs_ymarkup6','STRING',NULL,FALSE,TRUE),
  ('zipt_markup','trusted','ZIOMARKUP','fs_bic_fs_yiomarkup','STRING',NULL,TRUE,TRUE),
  ('zipt_markup','trusted','MKALK','fs_bic_fs_ymcstind','STRING',NULL,FALSE,TRUE)
  AS s(table_name, target_layer, source_column, target_column, target_data_type, transform_expr, is_business_key, is_active)
) AS s
ON t.table_name = s.table_name AND t.target_layer = s.target_layer AND t.source_column = s.source_column
WHEN MATCHED THEN UPDATE SET
  t.target_column    = s.target_column,
  t.target_data_type = s.target_data_type,
  t.transform_expr   = CAST(s.transform_expr AS STRING),
  t.is_business_key  = s.is_business_key,
  t.is_active        = s.is_active
WHEN NOT MATCHED THEN INSERT
  (table_name, target_layer, source_column, target_column, target_data_type, transform_expr, is_business_key, is_active)
VALUES
  (s.table_name, s.target_layer, s.source_column, s.target_column, s.target_data_type, CAST(s.transform_expr AS STRING), s.is_business_key, s.is_active)

In [0]:
%sql
-- Insert-only: an existing row keeps its watermark progress (NULL watermark = first load)
MERGE INTO IDENTIFIER(:catalog_name || '.metadata_ict.etl_watermark') AS t
USING (
  SELECT * FROM VALUES
    ('zipt_markup','raw','TIMESTAMP'),
    ('zipt_markup','foundation','_ingest_ts'), 
    ('zipt_markup','trusted','_ingest_ts'),
    ('zipt_markup','unified','_commit_version')
  AS s(entity_id, layer, watermark_column)
) AS s
ON t.entity_id = s.entity_id AND t.layer = s.layer
WHEN NOT MATCHED THEN INSERT
  (entity_id, layer, watermark_column, last_watermark_value, previous_watermark_value, last_run_id, last_load_type, last_success_ts, updated_by)
VALUES
  (s.entity_id, s.layer, s.watermark_column, NULL, NULL, NULL, NULL, NULL, NULL)